In [2]:
from ask_delphi_api.convert_taaksjabloon import read_dir, convert_doc_to_json
paths = read_dir('/Users/baasa03/Digicoach/taaksjabloon_demo')
json_coaches = []

for path in paths:
    try:
        print("\n\n")
        json_coach = convert_doc_to_json(path)
        json_coaches.append(json_coach)
    except ValueError as e:
        print(e)

LIC - Taaksjablonen Sanering v1.0.docx



retrieved 9 tables from doc /Users/baasa03/Digicoach/taaksjabloon_demo/LIC - Taaksjablonen Sanering v1.0.docx
Found title: Sanering (LIC)
Found tag: Directie CAP
Found tag: Keten Inning en betalingsverkeer
Found step: Open postbak in WAB
Found step: Selecteer alleen kwijtscheldings / saneringsverzoeken
Found step: In behandeling nemen poststuk saneringen
Found step: Controleer taken op je naam
Found task: Selecteer Zaak
Found step: Check op vermogensbestanddelen
Found step: Check op mogelijkheid aansprakelijk stellen
Found step: Check op teruggaven en goeder trouw
Found step: Check op ambtshalve aanslagen
Found step: Vastleggen acties
Found task: Quick scan verzoek
Found step: Controleer competentie
Found step: Controleer de datum van binnenkomst
Found step: Check op benodigde stukken
Found step: Check op onbehandelde zaken
Found step: Controleer volledigheid verzoek
Found step: Vastleggen brief of actie in INL
Found step: Acties bij verzoek is

In [3]:
print(json_coaches[0])
topic_id_list = []

{
  "name": "Sanering (LIC)",
  "tags": [
    {
      "type": "Directie",
      "values": [
        "CAP"
      ]
    },
    {
      "type": "Keten",
      "values": [
        "Inning en betalingsverkeer"
      ]
    }
  ],
  "tasks": [
    {
      "name": "Selecteer Zaak",
      "description": "In deze taak start je je werkzaamheden in WAB post op om de saneringsverzoeken te behandelen.",
      "steps": [
        {
          "name": "Open postbak in WAB",
          "description": "Ga naar de WAB-Post omgeving via de link https://wab.belastingdienst.nl/ProcessPortal"
        },
        {
          "name": "Selecteer alleen kwijtscheldings / saneringsverzoeken",
          "description": "Links in het grijze gedeelte druk je op de saved search \u2018\u2019Taskforce KWT/SAN\u2019\u2019, \u2018\u2019Taskforce KWT/SAN reactie ontvangen\u2019\u2019 en \u2018\u2019Taskforce KWT/SAN Reactie termijn afgelopen\u2019\u2019.  Op deze manier verschijnen de saneringsverzoeken in de werkomgeving.\n\n

In [ ]:
import json
from ask_delphi_api.import_digicoach import Import
import datetime

json_digicoach = json.loads(json_coaches[0])
naam_digicoach = f"Digicoach {json_digicoach['name']} {datetime.datetime.now()}"
tags= json_digicoach['tags']
name_voorgedefinieerde_zoekopdracht = ""
for tag in tags:
    if tag["type"] == "Keten" or tag["type"] == "Middel":
        name_voorgedefinieerde_zoekopdracht = tag["values"][0]

print(naam_digicoach)
print(name_voorgedefinieerde_zoekopdracht)


In [ ]:
from ask_delphi_api.import_digicoach import Import

import_service = Import()
topic_id_predefined_search, topic_version_id_predefined_search = import_service.create_voorgedefinieerde_zoekopdracht_topic(name_voorgedefinieerde_zoekopdracht)
topic_id_list.append(topic_id_predefined_search)

In [ ]:
# Maak Digicoach

topic_id_digicoach, topic_version_id_digicoach = import_service.create_digicoach(naam_digicoach, topic_id_predefined_search, topic_version_id_predefined_search)
topic_id_list.append(topic_id_digicoach)


In [ ]:
# Voeg tags toe
for tag in tags:
    # import_service.add_tag(topic_id_digicoach, topic_version_id_digicoach, tag["values"][0])
    import_service.add_tag(topic_id_digicoach, topic_version_id_digicoach, "interactie")

In [ ]:
import pprint
# Haal takenlijst uit json_digicoach
taken = json_digicoach["tasks"]
pprint.pp(taken[0])

# Voor elke taak, maak topic
for taak in taken:
    # Maak topic met naam taak uit json
    topic_id_task, topic_version_id_task = import_service.create_task(taak["name"], topic_id_digicoach, topic_version_id_digicoach)
    topic_id_list.append(topic_id_task)

    # Voeg beschrijving als content toe
    # import_service.add_content_to_topic(topic_id_task, topic_version_id_task, taak['description'])

    # voeg. stappen toe (later)
    # Haal stappenlijst uit taak
    steps = taak["steps"]
    # pprint.pp(steps[0])

    for step in steps:
    # Maak topic met naam step uit json
        topic_id_step, topic_version_id_step = import_service.create_step(step["name"], topic_id_task, topic_version_id_task)
        topic_id_list.append(topic_id_step)
        import_service._get_topic().checkin(topic_id_step)
    import_service._get_topic().checkin(topic_id_task)

import_service._get_topic().checkin(topic_id_digicoach)
import_service._get_topic().checkin(topic_id_predefined_search)

for topic_id in topic_id_list:
    import_service.publiceer(topic_id)



